<!-- DOC-BANNER -->
# Working with JSON, APIs and SQL (Day 16)

> **Playlist:** CampusX Day 16 *Working with JSON/SQL*.
> **Purpose:** Load the three non-CSV sources you will meet in ML: a local JSON file (`train.json`, 39774 rows of `id/cuisine/ingredients`), a REST API (GitHub issues for `ggplot2`, 30 rows x 36 cols), and MySQL (`world.country WHERE LifeExpectancy > 60`, 167 rows x 15 cols).
> **Prerequisites:** `pd.read_csv` (Day 15), basic JSON shape (objects/arrays).
> **After studying:** you can call `pd.read_json` on files and API URLs and `pd.read_sql_query` with a `mysql.connector` connection.


## Reading JSON files with `pd.read_json`

The recipe file loads straight into a DataFrame; nested lists (like `ingredients`) stay as lists inside cells until flattened.


In [1]:
import pandas as pd 

### 1. JSON file — `pd.read_json`
`pd.read_json('E:/MACHINE_LEARNING/train.json')` parses the recipe file directly into a DataFrame. Check shape/columns first: nested `ingredients` lists stay as Python lists inside cells (flatten later with `json_normalize`, see the [Day-16 deep-dive notebook](ML_DAY16_JSON-deep-dive.ipynb)).

In [3]:
df = pd.read_json(r"E:/MACHINE_LEARNING/train.json")
df

,id,cuisine,ingredients
0,10259,greek,"[romaine lettuce, black olives, grape tomatoes..."
1,25693,southern_us,"[plain flour, ground pepper, salt, tomatoes, g..."
2,20130,filipino,"[eggs, pepper, salt, mayonaise, cooking oil, g..."
3,22213,indian,"[water, vegetable oil, wheat, salt]"
4,13162,indian,"[black pepper, shallots, cornflour, cayenne pe..."
...,...,...,...
39769,29109,irish,"[light brown sugar, granulated sugar, butter, ..."
39770,11462,italian,"[KRAFT Zesty Italian Dressing, purple onion, b..."
39771,2238,irish,"[eggs, citrus fruit, raisins, sourdough starte..."
39772,41882,chinese,"[boneless chicken skinless thigh, minced garli..."


### 2. REST API — JSON over HTTP
`pd.read_json` can take a URL. The GitHub issues endpoint returns a list of issue objects → each becomes a row (36 columns). Real APIs need pagination/auth/rate-limits — this demo reads only the first page.

In [7]:
import pandas as pd

url = "https://api.github.com/repositories/19438/issues"

df = pd.read_json(url)

print(df)

                                                  url  \
0   https://api.github.com/repos/tidyverse/ggplot2...   
1   https://api.github.com/repos/tidyverse/ggplot2...   
2   https://api.github.com/repos/tidyverse/ggplot2...   
3   https://api.github.com/repos/tidyverse/ggplot2...   
4   https://api.github.com/repos/tidyverse/ggplot2...   
5   https://api.github.com/repos/tidyverse/ggplot2...   
6   https://api.github.com/repos/tidyverse/ggplot2...   
7   https://api.github.com/repos/tidyverse/ggplot2...   
8   https://api.github.com/repos/tidyverse/ggplot2...   
9   https://api.github.com/repos/tidyverse/ggplot2...   
10  https://api.github.com/repos/tidyverse/ggplot2...   
11  https://api.github.com/repos/tidyverse/ggplot2...   
12  https://api.github.com/repos/tidyverse/ggplot2...   
13  https://api.github.com/repos/tidyverse/ggplot2...   
14  https://api.github.com/repos/tidyverse/ggplot2...   
15  https://api.github.com/repos/tidyverse/ggplot2...   
16  https://api.github.com/repo

## Reading SQL tables with `pd.read_sql_query`

Send a SQL string through a database connection and get a DataFrame back.


### 3. SQL — `read_sql_query` via `mysql.connector`
`cc.connect(host='localhost', user='root', database='world')` opens MySQL, then `pd.read_sql_query('SELECT * FROM country WHERE LifeExpectancy>60', conn)` returns the filtered table. Close the connection when done; production code prefers SQLAlchemy (note the `UserWarning` below). This notebook's kernel crashed after the SQL cell — re-run needs a live MySQL server with the `world` database.

In [8]:
!pip install mysql.connector


     ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
      --------------------------------------- 0.3/11.9 MB ? eta -:--:--
      --------------------------------------- 0.3/11.9 MB ? eta -:--:--
     - ------------------------------------- 0.5/11.9 MB 833.6 kB/s eta 0:00:14
     -- ------------------------------------ 0.8/11.9 MB 775.7 kB/s eta 0:00:15
     -- ------------------------------------ 0.8/11.9 MB 775.7 kB/s eta 0:00:15
     -- ------------------------------------ 0.8/11.9 MB 775.7 kB/s eta 0:00:15
     --- ----------------------------------- 1.0/11.9 MB 673.2 kB/s eta 0:00:17
     --- ----------------------------------- 1.0/11.9 MB 673.2 kB/s eta 0:00:17
     ---- ---------------------------------- 1.3/11.9 MB 646.5 kB/s eta 0:00:17
     ------- -------------------------------- 2.4/11.9 MB 1.1 MB/s eta 0:00:09
     ----------- ---------------------------- 3.4/11.9 MB 1.5 MB/s eta 0:

In [ ]:
import mysql.connector as cc

conn = cc.connect(
    host='localhost',
    user='root',
    password='',
    database='world'
)
conn



df = pd.read_sql_query("SELECT * FROM country WHERE 	LifeExpectancy > 60 " , conn )

df 

C:\Users\saumi\AppData\Local\Temp\ipykernel_25700\2315980429.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM country WHERE 	LifeExpectancy > 60 " , conn )


,Code,Name,Continent,Region,SurfaceArea,IndepYear,Population,LifeExpectancy,GNP,GNPOld,LocalName,GovernmentForm,HeadOfState,Capital,Code2
0,ABW,Aruba,North America,Caribbean,193.0,NaN,103000,78.4,828.0,793.0,Aruba,Nonmetropolitan Territory of The Netherlands,Beatrix,129,AW
1,AIA,Anguilla,North America,Caribbean,96.0,NaN,8000,76.1,63.2,NaN,Anguilla,Dependent Territory of the UK,Elisabeth II,62,AI
2,ALB,Albania,Europe,Southern Europe,28748.0,1912.0,3401200,71.6,3205.0,2500.0,Shqipëria,Republic,Rexhep Mejdani,34,AL
3,AND,Andorra,Europe,Southern Europe,468.0,1278.0,78000,83.5,1630.0,NaN,Andorra,Parliamentary Coprincipality,,55,AD
4,ANT,Netherlands Antilles,North America,Caribbean,800.0,NaN,217000,74.7,1941.0,NaN,Nederlandse Antillen,Nonmetropolitan Territory of The Netherlands,Beatrix,33,AN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,VIR,"Virgin Islands, U.S.",North America,Caribbean,347.0,NaN,93000,78.1,0.0,NaN,Virgin Islands of the United States,US Territory,George W. Bush,4067,VI
163,VNM,Vietnam,Asia,Southeast Asia,331689.0,1945.0,79832000,69.3,21929.0,22834.0,Viêt Nam,Socialistic Republic,Trân Duc Luong,3770,VN
164,VUT,Vanuatu,Oceania,Melanesia,12189.0,1980.0,190000,60.6,261.0,246.0,Vanuatu,Republic,John Bani,3537,VU
165,WSM,Samoa,Oceania,Polynesia,2831.0,1962.0,180000,69.2,141.0,157.0,Samoa,Parlementary Monarchy,Malietoa Tanumafili II,3169,WS


: 

<!-- EXPERIMENT -->
### Experiment: change the question, change the table
Change the SQL filter to `LifeExpectancy > 70` and predict: will the row count grow or shrink, and why? Then re-read `data.jsonl` with `nrows=100` and check the shape — sampling first is the habit that keeps big-file exploration fast.

<!-- DOC-BANNER -->
## Key takeaways (Day 16)
- Three loaders, one output (DataFrame): `read_json` (file/URL) and `read_sql_query` (connection + SQL string).
- JSON `ingredients`-style columns are lists — inspect before modeling.
- **Common mistakes:** no `conn.close()`; no error handling if MySQL is offline; mixing JSON-file, API and SQL topics in one notebook makes revision harder (see `topic_notes.md` for separated summaries).
- **ML connection:** Data collection → inspection. Everything downstream (EDA → cleaning → split) assumes this step produced a correct DataFrame.
